# 1. Настройка окружения

**Цель:** Проверить установку PyTorch, определить доступные устройства (CPU/GPU/MPS), настроить логирование.

---

In [10]:
# Импортируем стандартные библиотеки Python
import sys
import logging
import os

# Читаем уровень логирования из переменной окружения LOG_LEVEL
# По умолчанию DEBUG — все сообщения, включая отладочные.
# Можно выставить: INFO, WARNING, ERROR, CRITICAL.
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")

# Настраиваем логирование: формат (время, уровень, имя логгера, сообщение)
# и поток вывода (stderr, чтобы не смешивать с stdout).
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    stream=sys.stderr,
)

# Создаём логгер с именем "setup" для этого ноутбука
log = logging.getLogger("setup")
log.info("Logging configured at %s level", LOG_LEVEL)


2026-05-31 19:17:21,921 [INFO] setup: Logging configured at DEBUG level


In [2]:
# Импортируем PyTorch — главный фреймворк для всех наших экспериментов
import torch

log.debug("Importing PyTorch")
print(f"PyTorch version: {torch.__version__}")
log.info("PyTorch %s loaded", torch.__version__)


2026-05-31 11:17:58,751 [DEBUG] setup: Importing PyTorch
2026-05-31 11:17:58,752 [INFO] setup: PyTorch 2.8.0 loaded


PyTorch version: 2.8.0


In [3]:
# Информация о системе: версия Python и платформа
import platform

print(f"Python: {sys.version}")           # версия интерпретатора
print(f"Platform: {platform.platform()}") # ОС и архитектура
print(f"Processor: {platform.processor()}")  # процессор
log.debug("System info gathered")


2026-05-31 11:18:04,859 [DEBUG] setup: System info gathered


Python: 3.9.6 (default, Apr 17 2026, 18:15:52) 
[Clang 21.0.0 (clang-2100.1.1.101)]
Platform: macOS-26.2-arm64-arm-64bit
Processor: arm


In [4]:
# === Проверка CUDA (NVIDIA GPU) ===
# CUDA позволяет запускать тензорные операции на видеокартах NVIDIA.
# torch.cuda.is_available() — проверяет, есть ли доступная CUDA-карта.
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    # Если CUDA есть — покажем имя GPU и версию CUDA toolkit
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    log.info("CUDA device: %s", torch.cuda.get_device_name(0))
else:
    log.warning("CUDA not available — either no NVIDIA GPU or no CUDA toolkit")


2026-05-31 11:18:14,821 [WARNING] setup: CUDA not available — either no NVIDIA GPU or no CUDA toolkit


CUDA available: False


In [5]:
# Проверка MPS (Apple Silicon)
mps_available = torch.backends.mps.is_available()
mps_built = torch.backends.mps.is_built()
print(f"MPS available: {mps_available}")
print(f"MPS built: {mps_built}")
if mps_available:
    log.info("MPS is available — will use Apple Silicon GPU")
else:
    log.warning("MPS not available")

2026-05-31 11:18:25,689 [INFO] setup: MPS is available — will use Apple Silicon GPU


MPS available: True
MPS built: True


## Почему важен выбор устройства (device)?

PyTorch поддерживает три типа устройств для вычислений:

| Устройство | Когда доступно | Скорость | Применение |
|------------|----------------|----------|------------|
| **CPU** | Всегда | Медленно | Отладка, маленькие модели |
| **CUDA** | NVIDIA GPU | ×10–100 быстрее CPU | Большие модели, batch-обработка |
| **MPS** | Apple Silicon (M1+) | ×5–20 быстрее CPU | То же, на Mac |

**Почему это важно:**
- Обучение трансформеров даже на маленьких датасетах занимает минуты на CPU и секунды на GPU
- В этом курсе мы будем использовать MPS (Apple Silicon) или CPU, если MPS недоступен
- Все тензоры нужно явно перемещать на устройство: `tensor.to(device)`
- Передача данных между CPU и GPU — дорогая операция (overhead), старайтесь держать данные на одном устройстве

Подробнее: [PyTorch MPS Documentation](https://pytorch.org/docs/stable/notes/mps.html)

In [6]:
# === Выбор устройства ===
# Приоритет: CUDA > MPS > CPU
# Это стандартный паттерн для всех PyTorch-проектов.
if torch.cuda.is_available():
    device = torch.device("cuda")       # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")        # Apple Silicon GPU
else:
    device = torch.device("cpu")        # Универсальное падение

print(f"Selected device: {device}")
log.info("Using device: %s", device)


2026-05-31 11:18:50,241 [INFO] setup: Using device: mps


Selected device: mps


In [7]:
# === Тестовый тензор ===
# torch.randn(3, 3) создаёт матрицу 3×3 из случайных чисел
# Параметр device=device перемещает тензор на выбранное устройство сразу при создании
# Это эффективнее, чем создать на CPU и потом .to(device)
x = torch.randn(3, 3, device=device)
print(f"Test tensor shape: {x.shape}")    # размерность (rows, cols)
print(f"Test tensor device: {x.device}")  # на каком устройстве лежит
print(f"Test tensor dtype: {x.dtype}")    # тип данных (float32 по умолчанию)
print(x)
log.debug("Test tensor created on %s: shape=%s", device, x.shape)


2026-05-31 11:19:02,897 [DEBUG] setup: Test tensor created on mps: shape=torch.Size([3, 3])


Test tensor shape: torch.Size([3, 3])
Test tensor device: mps:0
Test tensor dtype: torch.float32
tensor([[-0.8554,  1.8641, -0.8748],
        [ 0.7470,  0.9490, -1.5423],
        [ 0.0221, -0.6299,  1.7149]], device='mps:0')


In [8]:
# === Базовый benchmark: умножение матриц ===
# Это простой тест производительности вычислительного устройства.
# Матричное умножение — основа attention (Q @ K.T), поэтому скорость важна.
import time

sizes = [100, 1000, 5000]  # размеры матриц: от маленьких до больших
for n in sizes:
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    
    # Прогрев: первые запуски могут быть медленнее (JIT, выделение памяти)
    for _ in range(5):
        c = a @ b
    
    # На GPU операции асинхронные — нужно явно дождаться завершения
    # Иначе time.perf_counter() покажет неправильное (слишком маленькое) время
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    
    # Замер: 20 повторений и усреднение
    start = time.perf_counter()
    for _ in range(20):
        c = a @ b
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    elapsed = (time.perf_counter() - start) / 20
    
    # Результат: чем меньше время, тем быстрее устройство
    print(f"Matrix multiply {n}x{n}: {elapsed*1000:.2f} ms")
    log.info("Benchmark %dx%d: %.2f ms", n, n, elapsed * 1000)


2026-05-31 11:19:13,751 [INFO] setup: Benchmark 100x100: 0.07 ms
2026-05-31 11:19:13,782 [INFO] setup: Benchmark 1000x1000: 0.63 ms


Matrix multiply 100x100: 0.07 ms
Matrix multiply 1000x1000: 0.63 ms


2026-05-31 11:19:15,577 [INFO] setup: Benchmark 5000x5000: 63.06 ms


Matrix multiply 5000x5000: 63.06 ms


In [9]:
# === Итог ===
print("\n=== Environment check complete ===")
print(f"Summary: PyTorch {torch.__version__} on {device}")
log.info("Environment check complete — ready for transformer study")


2026-05-31 11:19:41,038 [INFO] setup: Environment check complete — ready for transformer study



=== Environment check complete ===
Summary: PyTorch 2.8.0 on mps
